# Neural Message Passing for Quantum Chemistry
## An executable reproduction of Gilmer et al. (2017)

This notebook explains the **Message Passing Neural Network (MPNN)** framework introduced by Justin Gilmer, Samuel S. Schoenholz, Patrick F. Riley, Oriol Vinyals, and George E. Dahl, and reproduces its central QM9 experiment in a compact form.

**Paper:** [Neural Message Passing for Quantum Chemistry](https://proceedings.mlr.press/v70/gilmer17a.html), ICML 2017 · [arXiv](https://arxiv.org/abs/1704.01212)

> Reproduction scope: the code implements the paper's best-performing **enn-s2s** family (edge-network messages + GRU updates + Set2Set readout). It now defaults to the paper-scale 110k/10k/10k split and a CUDA-oriented training configuration. Set `FULL_REPRODUCTION = False` only for a smoke test. Exact paper numbers are not expected: software, initialization, preprocessing, and random splits differ.

## 1. The paper's central idea

A molecule is a graph $G=(V,E)$: atoms are nodes, while pairwise relations are edges. Instead of hand-designing a fixed molecular fingerprint, an MPNN learns a representation in two phases.

### Message passing
For $T$ steps, every atom collects messages from other atoms and updates its hidden state:

$$m_v^{t+1}=\sum_{w\in\mathcal N(v)} M_t(h_v^t,h_w^t,e_{vw}),$$
$$h_v^{t+1}=U_t(h_v^t,m_v^{t+1}).$$

### Readout
A permutation-invariant function turns the set of final atom states into a molecular prediction:

$$\hat y=R(\{h_v^T\mid v\in G\}).$$

This abstraction unifies several earlier graph models. The paper compares message functions and readouts, finding that an **edge network** paired with **Set2Set** performs especially well on QM9.

## 2. The `enn-s2s` model reproduced here

The implementation below follows the important architectural choices from the paper:

1. **Complete graph:** every atom can message every other atom (no self-loops).
2. **Distance edge feature:** the Euclidean distance $d_{vw}=\|r_v-r_w\|_2$ is expanded in Gaussian radial basis functions.
3. **Edge network:** a neural network maps the edge feature to a full $H\times H$ matrix $A(e_{vw})$, giving
   $$m_v^{t+1}=\sum_w A(e_{vw})h_w^t.$$
4. **GRU update:** $h_v^{t+1}=\operatorname{GRU}(m_v^{t+1},h_v^t)$.
5. **Set2Set readout:** an LSTM repeatedly attends over atom states, yielding a permutation-invariant graph representation.

The paper predicts all 13 molecular properties with separate models. For a clean demonstration we predict **$U_0$**, internal energy at 0 K, and standardize the target using training-set statistics.

In [1]:
import copy
import math
import os
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch_geometric.data import Data
from torch_geometric.datasets import QM9
from torch_geometric.loader import DataLoader
from torch_geometric.nn import Set2Set
from torch_geometric.utils import scatter

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = device.type == 'cuda'
if USE_AMP:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision('high')
print(f'PyTorch {torch.__version__} | device: {device}')
if USE_AMP:
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name} | VRAM: {props.total_memory / 2**30:.1f} GiB | AMP: enabled')

PyTorch 2.12.0+cpu | device: cpu


## 3. QM9 and preprocessing

QM9 contains roughly 134k small organic molecules with DFT-computed quantum-chemical properties. PyG excludes uncharacterized molecules, leaving about 130k graphs. Node features include atom identity and basic chemical descriptors; `pos` contains 3D coordinates.

The original paper uses atom type as the initial state and interatomic distance as the edge input. We use the richer PyG node vector as input, but construct the same complete distance graph. This is one explicit difference from the paper.

In [2]:
class CompleteGraphDistance:
    """Replace chemical bonds by directed all-pairs edges with scalar distances."""
    def __call__(self, data):
        n = data.num_nodes
        row = torch.arange(n).repeat_interleave(n)
        col = torch.arange(n).repeat(n)
        keep = row != col
        # edge_index[0] is source and edge_index[1] is destination.
        data.edge_index = torch.stack([row[keep], col[keep]], dim=0)
        delta = data.pos[row[keep]] - data.pos[col[keep]]
        data.edge_attr = delta.norm(dim=-1, keepdim=True)
        return data

# Work both when Jupyter starts in the repository root and in MPNNs/.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'MPNNs' else Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data'
MODELS_DIR = PROJECT_ROOT / 'models'
DATA_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
DATA_ROOT = DATA_DIR / 'QM9'
dataset = QM9(root=DATA_ROOT, transform=CompleteGraphDistance())
sample = dataset[0]
print(f'Molecules: {len(dataset):,}')
print(sample)
print(f'Atoms: {sample.num_nodes}; complete directed edges: {sample.num_edges}')

Extracting c:\Users\Morit\Desktop\Projekte\pharmaceuticalAI\data\QM9\raw\qm9_v3.zip
Processing...
Using a pre-processed version of the dataset. Please install 'rdkit' to alternatively process the raw data.
Done!


Molecules: 130,831
Data(x=[5, 11], edge_index=[2, 20], edge_attr=[20, 1], y=[1, 19], pos=[5, 3], idx=[1], name='gdb_1', z=[5])
Atoms: 5; complete directed edges: 20


In [3]:
# PyG QM9 target columns. We reproduce one of the paper's regression tasks.
TARGETS = ['mu', 'alpha', 'HOMO', 'LUMO', 'gap', 'R2', 'ZPVE',
           'U0', 'U', 'H', 'G', 'Cv', 'U0_atom', 'U_atom', 'H_atom',
           'G_atom', 'A', 'B', 'C']
TARGET_NAME = 'U0'
TARGET_INDEX = TARGETS.index(TARGET_NAME)

FULL_REPRODUCTION = True
if FULL_REPRODUCTION:
    N_TRAIN, N_VAL, N_TEST = 110_000, 10_000, 10_000
    EPOCHS, BATCH_SIZE = 100, 128  # reduce to 64/32 if edge matrices exhaust VRAM
else:
    N_TRAIN, N_VAL, N_TEST = 8_000, 1_000, 1_000
    EPOCHS, BATCH_SIZE = 12, 64

g = torch.Generator().manual_seed(SEED)
perm = torch.randperm(len(dataset), generator=g)[:N_TRAIN + N_VAL + N_TEST]
train_set = dataset.index_select(perm[:N_TRAIN])
val_set = dataset.index_select(perm[N_TRAIN:N_TRAIN + N_VAL])
test_set = dataset.index_select(perm[N_TRAIN + N_VAL:])

# Read the contiguous backing target tensor so normalization does not build
# 110k complete graphs merely to access y. Statistics use training indices only.
train_targets = dataset._data.y[perm[:N_TRAIN], TARGET_INDEX]
target_mean = train_targets.mean()
target_std = train_targets.std().clamp_min(1e-8)

NUM_WORKERS = min(8, os.cpu_count() or 1) if device.type == 'cuda' else 0
loader_kwargs = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                     pin_memory=USE_AMP, persistent_workers=NUM_WORKERS > 0)
train_loader = DataLoader(train_set, shuffle=True, **loader_kwargs)
val_loader = DataLoader(val_set, **loader_kwargs)
test_loader = DataLoader(test_set, **loader_kwargs)
print(f'Split: {len(train_set):,}/{len(val_set):,}/{len(test_set):,}; batch={BATCH_SIZE}; workers={NUM_WORKERS}')
print(f'{TARGET_NAME}: mean={target_mean:.4f}, std={target_std:.4f}')

Split: 8,000/1,000/1,000
U0: mean=-11182.5586, std=1073.8470


## 4. Implementing the MPNN

`GaussianSmearing` turns one distance into a smooth vector. `EdgeNetworkConv` generates a different linear map for each edge and sums incoming messages. `MPNN` repeats this operation, updates nodes with a GRU, and applies Set2Set.

A useful invariance check: changing the order of atoms must not change the molecular prediction. Sum aggregation and Set2Set both operate on sets, so the architecture is invariant to node permutations (while still depending on the molecular geometry).

In [4]:
class GaussianSmearing(nn.Module):
    def __init__(self, start=0.0, stop=5.0, num_gaussians=50):
        super().__init__()
        offsets = torch.linspace(start, stop, num_gaussians)
        self.register_buffer('offsets', offsets)
        spacing = offsets[1] - offsets[0]
        self.coeff = -0.5 / float(spacing ** 2)

    def forward(self, distance):
        return torch.exp(self.coeff * (distance - self.offsets) ** 2)


class EdgeNetworkConv(nn.Module):
    def __init__(self, hidden_dim, edge_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.edge_network = nn.Sequential(
            nn.Linear(edge_dim, 128), nn.ReLU(),
            nn.Linear(128, hidden_dim * hidden_dim),
        )

    def forward(self, h, edge_index, edge_features):
        src, dst = edge_index
        matrices = self.edge_network(edge_features).view(-1, self.hidden_dim, self.hidden_dim)
        messages = torch.bmm(matrices, h[src].unsqueeze(-1)).squeeze(-1)
        return scatter(messages, dst, dim=0, dim_size=h.size(0), reduce='sum')


class MPNN(nn.Module):
    def __init__(self, node_dim, hidden_dim=64, edge_dim=50, steps=3, set2set_steps=3):
        super().__init__()
        self.steps = steps
        self.distance_expansion = GaussianSmearing(num_gaussians=edge_dim)
        self.node_encoder = nn.Linear(node_dim, hidden_dim)
        self.message = EdgeNetworkConv(hidden_dim, edge_dim)
        self.update = nn.GRUCell(hidden_dim, hidden_dim)
        self.readout = Set2Set(hidden_dim, processing_steps=set2set_steps)
        self.predictor = nn.Sequential(
            nn.Linear(2 * hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, data):
        h = self.node_encoder(data.x)
        edge_features = self.distance_expansion(data.edge_attr)
        for _ in range(self.steps):
            m = self.message(h, data.edge_index, edge_features)
            h = self.update(m, h)
        graph_embedding = self.readout(h, data.batch)
        return self.predictor(graph_embedding).squeeze(-1)

model = MPNN(node_dim=dataset.num_node_features).to(device)
n_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
batch = next(iter(train_loader)).to(device)
with torch.no_grad():
    output = model(batch)
print(model)
print(f'Parameters: {n_parameters:,}; output shape: {tuple(output.shape)}')

MPNN(
  (distance_expansion): GaussianSmearing()
  (node_encoder): Linear(in_features=11, out_features=64, bias=True)
  (message): EdgeNetworkConv(
    (edge_network): Sequential(
      (0): Linear(in_features=50, out_features=128, bias=True)
      (1): ReLU()
      (2): Linear(in_features=128, out_features=4096, bias=True)
    )
  )
  (update): GRUCell(64, 64)
  (readout): Set2Set(64, 128)
  (predictor): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=1, bias=True)
  )
)
Parameters: 618,625; output shape: (64,)


In [5]:
# Sanity check: two identical graphs in a batch receive identical predictions.
# (A direct node permutation is more cumbersome because all aligned tensors must be remapped.)
from torch_geometric.data import Batch
model.eval()
twins = Batch.from_data_list([dataset[0], copy.deepcopy(dataset[0])]).to(device)
with torch.no_grad():
    twin_predictions = model(twins)
print('Twin predictions:', twin_predictions.cpu().tolist())
assert torch.allclose(twin_predictions[0], twin_predictions[1], atol=1e-6)

Twin predictions: [-0.017312362790107727, -0.017312362790107727]


## 5. Training and evaluation

We optimize mean squared error on standardized targets, select the checkpoint with the lowest validation MAE, and report test MAE in the original PyG units. Gradient clipping helps stabilize the matrix-valued edge network.

In [6]:
mean = target_mean.to(device)
std = target_std.to(device)

def standardized_target(batch):
    return (batch.y[:, TARGET_INDEX] - mean) / std

def train_epoch(model, loader, optimizer):
    model.train()
    total_squared_error = 0.0
    total_graphs = 0
    for batch in loader:
        batch = batch.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            prediction = model(batch)
            target = standardized_target(batch)
            loss = nn.functional.mse_loss(prediction, target)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        scaler.step(optimizer)
        scaler.update()
        total_squared_error += loss.item() * batch.num_graphs
        total_graphs += batch.num_graphs
    return total_squared_error / total_graphs

@torch.no_grad()
def evaluate(model, loader, return_values=False):
    model.eval()
    predictions, targets = [], []
    for batch in loader:
        batch = batch.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            pred = model(batch) * std + mean
        predictions.append(pred.cpu())
        targets.append(batch.y[:, TARGET_INDEX].cpu())
    predictions = torch.cat(predictions)
    targets = torch.cat(targets)
    mae = (predictions - targets).abs().mean().item()
    return (mae, predictions, targets) if return_values else mae

In [ ]:
FORCE_RETRAIN = False
CHECKPOINT_PATH = MODELS_DIR / f'mpnn_qm9_{TARGET_NAME}_best.pt'
history = {'train_mse': [], 'val_mae': []}
best_val, start = float('inf'), time.time()

if CHECKPOINT_PATH.exists() and not FORCE_RETRAIN:
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=True)
    compatible = (checkpoint['target_name'] == TARGET_NAME and
                  checkpoint['node_dim'] == dataset.num_node_features and
                  checkpoint['split'] == (N_TRAIN, N_VAL, N_TEST, SEED))
    if not compatible:
        raise ValueError('Incompatible MPNN checkpoint; set FORCE_RETRAIN=True to replace it.')
    model.load_state_dict(checkpoint['model_state'])
    best_val = checkpoint['best_val']
    print(f'Loaded reusable checkpoint: {CHECKPOINT_PATH}')
else:
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-6, fused=USE_AMP)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.7, patience=5)
    stale_epochs, patience = 0, 20
    for epoch in range(1, EPOCHS + 1):
        train_mse = train_epoch(model, train_loader, optimizer)
        val_mae = evaluate(model, val_loader)
        scheduler.step(val_mae)
        history['train_mse'].append(train_mse)
        history['val_mae'].append(val_mae)
        if val_mae < best_val:
            best_val, stale_epochs = val_mae, 0
            torch.save({'model_state': model.state_dict(), 'target_name': TARGET_NAME,
                        'node_dim': dataset.num_node_features, 'best_val': best_val,
                        'split': (N_TRAIN, N_VAL, N_TEST, SEED),
                        'target_mean': target_mean.item(), 'target_std': target_std.item()}, CHECKPOINT_PATH)
        else:
            stale_epochs += 1
        print(f'Epoch {epoch:03d} | train MSE(z) {train_mse:.5f} | val MAE {val_mae:.5f} | lr {optimizer.param_groups[0]["lr"]:.2e}')
        if stale_epochs >= patience:
            print(f'Early stopping after {patience} epochs without improvement.')
            break
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=True)
    model.load_state_dict(checkpoint['model_state'])
test_mae, test_predictions, test_targets = evaluate(model, test_loader, return_values=True)
print(f'Best validation MAE: {best_val:.6f}')
print(f'Test MAE ({TARGET_NAME}): {test_mae:.6f}')
print(f'Elapsed: {(time.time() - start) / 60:.1f} min')

Epoch 001 | train MSE(z) 0.60230 | val MAE 456.99783
Epoch 002 | train MSE(z) 0.28072 | val MAE 705.57196
Epoch 003 | train MSE(z) 0.15281 | val MAE 177.49178


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['train_mse'], label='train MSE (standardized)')
axes[0].plot(history['val_mae'], label='validation MAE (original units)')
axes[0].set(xlabel='epoch', title='Learning curves')
axes[0].legend()

axes[1].scatter(test_targets.numpy(), test_predictions.numpy(), s=9, alpha=0.45)
lo = min(test_targets.min().item(), test_predictions.min().item())
hi = max(test_targets.max().item(), test_predictions.max().item())
axes[1].plot([lo, hi], [lo, hi], 'k--', linewidth=1)
axes[1].set(xlabel='true value', ylabel='predicted value', title=f'{TARGET_NAME}: test predictions')
plt.tight_layout()
plt.show()

## 6. Reading the result critically

A successful run should show falling validation error and predictions clustering around the diagonal. The number itself is less important than understanding what the experiment establishes: a learned, permutation-invariant molecular representation can be trained end-to-end from atoms and geometry.

### Why this is a reproduction, not an exact replication

- `FULL_REPRODUCTION=True` uses the paper-scale 110k/10k/10k split; the exact molecule assignment may differ.
- We use PyG's node features rather than only an atom-type encoding.
- We demonstrate one target; the paper trains and reports all 13 QM9 targets.
- The CUDA/AMP implementation and optimizer differ from the original training stack.
- The paper reports extensive model comparisons and multiple runs; this notebook trains one model once.

### Useful extensions

1. Loop over the 13 non-atomic QM9 targets and compare normalized MAE.
2. Replace Set2Set with sum pooling to isolate the readout's contribution.
3. Replace the edge network with a simple MLP message and perform an ablation.
4. Use only atom identity for a closer match to the paper.
5. Run several seeds and report mean ± standard deviation.

## References

- Gilmer et al. (2017), *Neural Message Passing for Quantum Chemistry*, Proceedings of ICML 70:1263–1272.
- Vinyals, Bengio & Kudlur (2016), *Order Matters: Sequence to Sequence for Sets* (Set2Set).
- Ramakrishnan et al. (2014), *Quantum chemistry structures and properties of 134 kilo molecules* (QM9).
- PyTorch Geometric [`QM9`](https://pytorch-geometric.readthedocs.io/en/latest/generated/torch_geometric.datasets.QM9.html) and [`Set2Set`](https://pytorch-geometric.readthedocs.io/en/latest/generated/torch_geometric.nn.aggr.Set2Set.html) documentation.